# Part 9 · Agentdesktop

**The story:** the gateway demos so far all begin after the request has arrived. Something on a laptop decided to send it, with some credential, from some tool nobody centrally configured. This part covers that side.

A daemon on each workstation enrols with a controller, reports which AI tools are installed and what they are wired to, applies the configuration the platform team publishes, and fetches a short-lived credential for the gateway instead of holding a provider key. Offboarding stops being a key hunt: disable the account and access ends.

<div align="center"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 800 250" style="width:100%;max-width:900px;height:auto" font-family="-apple-system,Segoe UI,Roboto,sans-serif">
  <rect x="0" y="0" width="800" height="250" rx="10" fill="#f8fafc"/>
  <text x="400" y="26" text-anchor="middle" font-size="14" font-weight="700" fill="#0f172a">The key lives at the gateway, the identity comes from the IdP</text>

  <rect x="325" y="48" width="150" height="48" rx="6" fill="#ede9fe" stroke="#7c3aed"/>
  <text x="400" y="70" text-anchor="middle" font-size="12" font-weight="700" fill="#4c1d95">Keycloak</text>
  <text x="400" y="86" text-anchor="middle" font-size="10" fill="#4c1d95">corp realm</text>

  <rect x="30" y="150" width="150" height="62" rx="6" fill="#e0f2fe" stroke="#0284c7"/>
  <text x="105" y="176" text-anchor="middle" font-size="12" font-weight="700" fill="#075985">Workstation</text>
  <text x="105" y="194" text-anchor="middle" font-size="10" fill="#075985">daemon + Claude Code</text>

  <rect x="255" y="150" width="150" height="62" rx="6" fill="#ede9fe" stroke="#7c3aed"/>
  <text x="330" y="176" text-anchor="middle" font-size="12" font-weight="700" fill="#4c1d95">Controller</text>
  <text x="330" y="194" text-anchor="middle" font-size="10" fill="#4c1d95">policy + JWT</text>

  <rect x="480" y="150" width="150" height="62" rx="6" fill="#dcfce7" stroke="#16a34a"/>
  <text x="555" y="176" text-anchor="middle" font-size="12" font-weight="700" fill="#14532d">agentgateway</text>
  <text x="555" y="194" text-anchor="middle" font-size="10" fill="#14532d">holds the API key</text>

  <rect x="700" y="150" width="80" height="62" rx="6" fill="#fef3c7" stroke="#d97706"/>
  <text x="740" y="176" text-anchor="middle" font-size="12" font-weight="700" fill="#78350f">Anthropic</text>
  <text x="740" y="194" text-anchor="middle" font-size="10" fill="#78350f">frontier</text>

  <line x1="180" y1="181" x2="255" y2="181" stroke="#334155" stroke-width="1.5" marker-end="url(#a9)"/>
  <text x="217" y="174" text-anchor="middle" font-size="9" fill="#334155">enrol</text>

  <line x1="405" y1="181" x2="480" y2="181" stroke="#334155" stroke-width="1.5" marker-end="url(#a9)"/>
  <text x="442" y="174" text-anchor="middle" font-size="9" fill="#334155">JWT 5m</text>

  <line x1="630" y1="181" x2="700" y2="181" stroke="#334155" stroke-width="1.5" marker-end="url(#a9)"/>
  <text x="665" y="174" text-anchor="middle" font-size="9" fill="#334155">key</text>

  <line x1="330" y1="150" x2="330" y2="96" stroke="#334155" stroke-width="1.5" marker-end="url(#a9)"/>
  <text x="352" y="126" text-anchor="start" font-size="9" fill="#334155">verify user</text>

  <defs><marker id="a9" markerWidth="9" markerHeight="9" refX="8" refY="3" orient="auto">
    <path d="M0,0 L0,6 L9,3 z" fill="#334155"/></marker></defs>
</svg></div>

Part 7 built the gateway this uses, so run its standup first.

## Reset

Puts the demo back to a clean state. Removes the fleet and the controller, and leaves Part 7's gateway up. Run it before a fresh demo.


In [ ]:
cd "$(git rev-parse --show-toplevel)/vision-demo-2026"
docker rm -f ws-london ws-berlin ws-madrid ws-dublin 2>/dev/null
./demo-scripts/agentdesktop.sh teardown

## Connect

Every other cell assumes this has run.

In [ ]:
cd "$(git rev-parse --show-toplevel)/vision-demo-2026"
export CTX=kind-mesh1
kubectl config use-context $CTX
# written by agentdesktop.sh: controller, Keycloak and gateway addresses
[ -f demo-scripts/.agentdesktop-env ] && . demo-scripts/.agentdesktop-env
echo "controller=$AD_CONTROLLER_IP  keycloak=$AD_KEYCLOAK_IP  gateway=$AD_GATEWAY" 

## 1. Stand up the platform

One script. It adds a `corp` realm to the Keycloak already running on mesh1, gives it a public `agentdesktop` client and three staff accounts, then deploys PostgreSQL, the controller and its TLS material.


In [ ]:
./demo-scripts/agentdesktop.sh

The controller serves two things. Port 443 is the device-facing fleet API and the JWKS the gateway reads. The console is on loopback inside the pod, so reach it with a port-forward.

In [ ]:
kubectl -n agentdesktop port-forward deploy/agentdesktop 18099:8080 >/tmp/ad-pf.log 2>&1 &
sleep 5
curl -sS http://127.0.0.1:18099/api/v1/overview | jq

## 2. Enrol this laptop

Enrolment is a certificate signing request. The daemon generates a private key locally, sends the CSR, signs in through Keycloak, and the controller issues a device certificate bound to the account that signed in. The key does not leave the machine.

The controller certificate is issued for its in-cluster DNS name and the OIDC issuer uses the same form of name, so the laptop has to resolve both the way the cluster does.

The daemon reconciles into Claude Code's real `settings.json`. If another gateway demo is currently routing Claude, `up` will stop and say so: turn that one off first. `AD_SAFE=1` sends the managed file to `/tmp` instead.

`./demo-scripts/agentdesktop-enrol-mac.sh state` says which demo owns Claude, and `down` unenrols and puts it back.


In [ ]:
./demo-scripts/agentdesktop-enrol-mac.sh binary   # download + sign the device binary
./demo-scripts/agentdesktop-enrol-mac.sh hosts    # the two /etc/hosts lines
# add them with sudo, then:
./demo-scripts/agentdesktop-enrol-mac.sh preview


Bring it up in a terminal. It stays in the foreground and opens a browser; sign in as `tom` / `password`.

```
./demo-scripts/agentdesktop-enrol-mac.sh up
```

Then from here:


In [ ]:
./demo-scripts/agentdesktop-enrol-mac.sh status


### Both clients

The daemon above runs as you, and manages Claude Code alone. Claude Desktop reads its
policy from the machine's managed preferences, which only a daemon running as the
machine can write, and a `--user` daemon refuses a policy carrying `programs.claudeDesktop`
**whole**, so Claude Code ends up unmanaged with it.

`install-daemon` registers Agentdesktop at the path the product itself looks for,
`/Library/LaunchDaemons/dev.agentdesktop.daemon.plist`. One macOS authorisation, then
both clients are policy. `signin-url` prints the daemon's own sign-in page.


In [ ]:
./demo-scripts/agentdesktop-enrol-mac.sh install-daemon   # one macOS password prompt
./demo-scripts/agentdesktop-enrol-mac.sh signin-url       # open it, sign in as bob / password
./demo-scripts/agentdesktop-enrol-mac.sh daemon-state
# publish the both-clients policy, then:
#   ./demo-scripts/agentdesktop-enrol-mac.sh remove-daemon


### The rest of the fleet

One machine does not show much about a fleet. `agentdesktop-fleet/` builds an image from the published Linux binary and brings up four more against the same controller, each with a different account and a different set of MCP servers, so the console has an estate to show.


In [ ]:
cd demo-scripts/agentdesktop-fleet
cp ~/code/solo/agentdesktop/bin/agentdesktop-linux-arm64 . 2>/dev/null
docker build -q -t agentdesktop-workstation:latest .
cd ../..

In [ ]:
# AD_SYSTEM=1 runs the daemon as the machine rather than as one user. It is the only
# mode that can manage Claude Desktop, and it accepts a Claude Code-only policy just the
# same, so the fleet comes up this way whichever policy is published. A --user device
# refuses a policy carrying programs.claudeDesktop outright, and refuses the whole
# revision, so Claude Code would go unmanaged with it.
launch() { # name hostname user mcp-json skills
  docker rm -f "$1" >/dev/null 2>&1
  docker run -d --name "$1" --hostname "$2" --network kind \
    -v /tmp/agentdesktop-device-ca.pem:/etc/agentdesktop/device-ca.pem:ro \
    -e AD_SYSTEM=1 \
    -e AD_CONTROLLER_IP=$AD_CONTROLLER_IP -e AD_KEYCLOAK_IP=$AD_KEYCLOAK_IP \
    -e AD_USER="$3" -e AD_PASSWORD=password -e AD_MCP_JSON="$4" -e AD_SKILLS="$5" \
    agentdesktop-workstation:latest >/dev/null
}
launch ws-london london-mbp tom \
  '{"mcpServers":{"github":{"type":"http","url":"https://api.githubcopilot.com/mcp/"},"postgres":{"command":"npx","args":["-y","@modelcontextprotocol/server-postgres"]}}}' \
  'code-review release-notes'
launch ws-berlin berlin-tp priya \
  '{"mcpServers":{"github":{"type":"http","url":"https://api.githubcopilot.com/mcp/"},"jira":{"command":"npx","args":["-y","jira-mcp"]},"random-gist-tool":{"command":"node","args":["/home/dev/gist/index.js"]}}}' \
  'triage'
launch ws-madrid madrid-mbp leaver \
  '{"mcpServers":{"github":{"type":"http","url":"https://api.githubcopilot.com/mcp/"},"internal-wiki":{"command":"npx","args":["-y","wiki-mcp"]}}}' 'triage'
launch ws-dublin dublin-mbp tom \
  '{"mcpServers":{"slack":{"command":"npx","args":["-y","slack-mcp"]},"filesystem":{"command":"npx","args":["-y","@modelcontextprotocol/server-filesystem"]}}}' \
  'code-review'
sleep 70
for c in ws-london ws-berlin ws-madrid ws-dublin; do
  printf '%-11s ' "$c"; docker logs "$c" 2>&1 | grep -oE 'enrolled|did not complete' | tail -1
done

## 3. What the fleet view shows

Which machines are connected, which AI tools are on them, and whether the policy actually applied.


In [ ]:
curl -sS http://127.0.0.1:18099/api/v1/overview | jq '{total_devices,online_devices,config_failures,active_revision}'

In [ ]:
curl -sS http://127.0.0.1:18099/api/v1/devices | jq -r '
  .[] | "\(.hostname)  rev=\(.config_revision)  tools=\(.installed_tools|join(","))  by=\(.enrolled_by_subject[0:8])"'

Open `http://127.0.0.1:18099/` for the console.

![Fleet overview](img/fleet-overview.png)

The inventory per device is the part a security team wants, because an MCP server is third-party software wired into an agent without review. Note what it records and what it does not: the name, the transport and the source file, but not the command arguments, the environment variables or the HTTP headers. Those are where credentials sit.

In [ ]:
DID=$(curl -sS http://127.0.0.1:18099/api/v1/devices | jq -r '.[]|select(.hostname=="berlin-tp").id')
curl -sS http://127.0.0.1:18099/api/v1/devices/$DID | jq '.discoveries[] | {kind, mcp: [.mcp_servers[].name], skills: (.skills|length)}'

![Device inventory](img/device-inventory.png)

## 4. Changing policy for everyone

The daemon config on the workstation names the controller and nothing else. Everything a developer actually experiences, which gateway their tools use, what the sandbox permits, what Claude Code is told, arrives as a numbered revision.

The console builds it and shows the YAML it produces.

![Central configuration](img/central-config.png)

Sandbox intent is declared once and written in each tool's own format. Claude Code takes JSON with separate read and write deny lists; Codex takes TOML that subtracts from a baseline. Neither tool needs to know Agentdesktop wrote the file.

In [ ]:
# Every device reports the revision it reconciled, so a failed rollout is visible
curl -sS http://127.0.0.1:18099/api/v1/devices | jq -r '.[] | "\(.hostname)  revision=\(.config_revision)  error=\(if .config_error=="" then "none" else .config_error end)"'

## 5. The credential on the laptop

The daemon merged into the existing settings rather than replacing them. Three things arrived without anyone touching this machine: a base URL pointing at the corporate gateway, an `apiKeyHelper` that calls back into the daemon, and the sandbox.


In [ ]:
./demo-scripts/agentdesktop-enrol-mac.sh settings


`CLAUDE_CODE_API_KEY_HELPER_TTL_MS` is how often Claude Code re-runs the helper, not how long the credential lasts.

Ask the daemon for a credential the way that helper does:


In [ ]:
docker exec ws-london sh -c 'agentdesktop credential --client-id claude-code' \
  | cut -d. -f2 | tr '_-' '/+' | base64 -d 2>/dev/null | jq '{iss,aud,email,client_id,lifetime:(.exp-.iat)}'

Five minutes, issued by the controller, carrying the user and the tool. Send it at the gateway.

In [ ]:
ASK='{"model":"claude-haiku-4-5","max_tokens":40,"messages":[{"role":"user","content":"Reply with exactly: the gateway holds the key"}]}'
docker exec -e GW="$AD_GATEWAY" -e ASK="$ASK" ws-london sh -c '
  TOK=$(agentdesktop credential --client-id claude-code)
  curl -sS http://$GW/v1/messages \
    -H "Authorization: Bearer $TOK" \
    -H "content-type: application/json" -H "anthropic-version: 2023-06-01" \
    -d "$ASK"' | jq '{model, content, input:.usage.input_tokens, output:.usage.output_tokens}'

## 6. Nothing else gets through

The gateway runs strict JWT. A developer cannot skip the daemon and call it directly, and the key they would need to skip the gateway is not on their machine.

In [ ]:
echo -n 'no credential:  '; curl -sS -o /dev/null -w 'HTTP %{http_code}\n' http://$AD_GATEWAY/v1/messages \
  -H 'content-type: application/json' -H 'anthropic-version: 2023-06-01' \
  -d '{"model":"claude-haiku-4-5","max_tokens":10,"messages":[{"role":"user","content":"hi"}]}'
echo -n 'forged token:   '; curl -sS -o /dev/null -w 'HTTP %{http_code}\n' http://$AD_GATEWAY/v1/messages \
  -H 'Authorization: Bearer not.a.jwt' \
  -H 'content-type: application/json' -H 'anthropic-version: 2023-06-01' \
  -d '{"model":"claude-haiku-4-5","max_tokens":10,"messages":[{"role":"user","content":"hi"}]}'

## 7. Someone leaves

The usual version of this is a key rotation, because a provider key has been copied into an unknown number of places and none of them are listed anywhere.

Here the laptop never had the key. Disable the account and revoke the sessions, which is what an offboarding process already does.

In [ ]:
KC_POD=$(kubectl -n keycloak get pod -l app=keycloak -o jsonpath='{.items[0].metadata.name}')
kcadm() { kubectl -n keycloak exec $KC_POD -- /opt/keycloak/bin/kcadm.sh "$@"; }
kcadm config credentials --server http://localhost:8080 --realm master --user admin --password admin >/dev/null
LID=$(kcadm get users -r corp -q username=leaver --fields id --format csv --noquotes | tr -d '\r')
kcadm update users/$LID -r corp -s enabled=false
kcadm create users/$LID/logout -r corp
echo "leaver disabled"


In [ ]:
HI='{"model":"claude-haiku-4-5","max_tokens":10,"messages":[{"role":"user","content":"hi"}]}'
echo '--- the leaver ---'
docker exec ws-madrid sh -c 'agentdesktop credential --client-id claude-code' 2>&1 | tail -2
echo
echo '--- everybody else ---'
for c in ws-london ws-berlin ws-dublin; do
  printf '%-11s ' "$c"
  docker exec -e GW="$AD_GATEWAY" -e HI="$HI" $c sh -c '
    TOK=$(agentdesktop credential --client-id claude-code 2>/dev/null)
    EMAIL=$(echo "$TOK" | cut -d. -f2 | tr "_-" "/+" | base64 -d 2>/dev/null | jq -r .email)
    CODE=$(curl -sS -o /dev/null -w "%{http_code}" http://$GW/v1/messages \
      -H "Authorization: Bearer $TOK" -H "content-type: application/json" \
      -H "anthropic-version: 2023-06-01" -d "$HI")
    echo "$EMAIL -> HTTP $CODE"'
done

The refresh fails as soon as the session is gone, so the credential is refused outright. Where it does not, the exposure is bounded by `gatewayJwt.lifetime`, which is five minutes here and is a number you choose centrally rather than a key you have to chase.

An offboarded account cannot enrol a new machine either, which is worth showing if anyone asks.

Put them back for the next run:

In [ ]:
kcadm update users/$LID -r corp -s enabled=true && echo "leaver re-enabled"